# Email Triage Agent — Idempotency Demo & Proof

This notebook runs `email_triage_agent.py`, an agent that fetches emails, classifies them, takes an action (reply / create task / archive / escalate), and — the important part — records what it did so that **running it again never creates duplicates**.

Full design rationale is in `README.md` and the docstrings/comments throughout `email_triage_agent.py`. This notebook is the runnable, timestamped proof: every run below prints a real wall-clock timestamp, and Colab preserves each cell's output once it's been run, so this notebook (downloaded afterward with outputs intact) is a verifiable record of what actually happened and when.

**Before running:** have your project folder (containing `email_triage_agent.py`, `test_agent.py`, and `test_mailtm_source.py`, all flat, no subfolder) uploaded to your Google Drive. Step 1 below mounts that Drive folder directly, so nothing needs to be uploaded through Colab's file picker.

## Step 1 — Connect to your Google Drive folder

Since your project folder is already in Google Drive, mount it here instead of re-uploading files each session. The first run of the next cell prompts you to authorize access (a Google sign-in popup) -- after that, your files are available for the rest of this session, and state persists across disconnects too, since it saves right next to the script in Drive rather than Colab's temporary storage.

**Update `PROJECT_DIR` below** to match wherever this folder actually lives in your Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/lec"  # <-- update this path
os.chdir(PROJECT_DIR)

print("Working in:", os.getcwd())
print("Files found:", sorted(os.listdir(".")))
assert "email_triage_agent.py" in os.listdir("."), "email_triage_agent.py not found -- check PROJECT_DIR above"


Mounted at /content/drive
Working in: /content/drive/MyDrive/lec
Files found: ['README.md', 'demo_colab.ipynb', 'email_triage_agent.py', 'test_agent.py', 'test_mailtm_source.py']


**One thing to know:** since state now lives on your Drive-mounted path by default, not Colab's local disk, there's a small chance you'll hit a `disk I/O error` on write -- some network-backed filesystems don't fully support SQLite's file locking (this project hit exactly that once before with a different mounted folder). If it happens, override `STATE_DIR` to a local Colab path instead:

```python
os.environ["STATE_DIR"] = "/content/state"
```

This doesn't affect the pytest suite in Step 4 -- those tests always use their own isolated temporary directories regardless of where this notebook's working directory points, so they're unaffected either way.

In [2]:
import subprocess, sys, os

SCRIPT = "email_triage_agent.py"

def run_agent(cmd, env_overrides=None):
    # Runs `python email_triage_agent.py <cmd>` as a real subprocess.
    # env_overrides only applies to this one call -- it can't leak into
    # later cells the way %env would.
    env = os.environ.copy()
    if env_overrides:
        env.update(env_overrides)
    result = subprocess.run([sys.executable, SCRIPT, cmd], env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print("--- stderr ---")
        print(result.stderr)
    return result


## Step 2 — Basic demo: prove no duplicates (stub inbox)

Run once: everything gets newly processed. Run again with no reset: zero new tasks, zero new replies, zero re-handled emails. Each printed summary below includes a real timestamp.

In [3]:
run_agent("reset")

State wiped.



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'reset'], returncode=0, stdout='State wiped.\n', stderr='')

In [4]:
run_agent("run")  # first run -- everything newly processed


=== Run dadc050b summary ===
  Timestamp:             2026-08-18T17:15:21.370060+00:00
  Source:                stub
  Classifier:            heuristic
  Emails fetched:        6
  Newly processed:       6
  Skipped (dup-guard):   0
    - msg-001: replied (reply)
    - msg-002: archived (archive)
    - msg-003: escalated (escalate)
    - msg-004: replied (reply)
    - msg-005: task_created (create_task)
    - msg-006: archived (archive)



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='\n=== Run dadc050b summary ===\n  Timestamp:             2026-08-18T17:15:21.370060+00:00\n  Source:                stub\n  Classifier:            heuristic\n  Emails fetched:        6\n  Newly processed:       6\n  Skipped (dup-guard):   0\n    - msg-001: replied (reply)\n    - msg-002: archived (archive)\n    - msg-003: escalated (escalate)\n    - msg-004: replied (reply)\n    - msg-005: task_created (create_task)\n    - msg-006: archived (archive)\n', stderr='')

In [5]:
run_agent("run")  # second run, same inbox -- expect "Newly processed: 0"


=== Run b31e2eab summary ===
  Timestamp:             2026-08-18T17:15:27.107109+00:00
  Source:                stub
  Classifier:            heuristic
  Emails fetched:        6
  Newly processed:       0
  Skipped (dup-guard):   6
    - msg-001: skipped_already_handled
    - msg-002: skipped_already_handled
    - msg-003: skipped_already_handled
    - msg-004: skipped_already_handled
    - msg-005: skipped_already_handled
    - msg-006: skipped_already_handled



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='\n=== Run b31e2eab summary ===\n  Timestamp:             2026-08-18T17:15:27.107109+00:00\n  Source:                stub\n  Classifier:            heuristic\n  Emails fetched:        6\n  Newly processed:       0\n  Skipped (dup-guard):   6\n    - msg-001: skipped_already_handled\n    - msg-002: skipped_already_handled\n    - msg-003: skipped_already_handled\n    - msg-004: skipped_already_handled\n    - msg-005: skipped_already_handled\n    - msg-006: skipped_already_handled\n', stderr='')

## Step 3 — Inspect stored state

In [6]:
run_agent("show")


-- emails --
{'id': 'msg-001', 'sender': 'alice@customer.com', 'subject': 'Can you send me the Q3 invoice?', 'category': 'reply', 'action': 'reply', 'status': 'handled'}
{'id': 'msg-002', 'sender': 'no-reply@newsletter.io', 'subject': 'Your weekly digest is here', 'category': 'archive', 'action': 'archive', 'status': 'handled'}
{'id': 'msg-003', 'sender': 'ops@vendor-systems.com', 'subject': 'URGENT: production outage affecting billing', 'category': 'escalate', 'action': 'escalate', 'status': 'handled'}
{'id': 'msg-004', 'sender': 'bob@partner.org', 'subject': 'Question about API rate limits', 'category': 'reply', 'action': 'reply', 'status': 'handled'}
{'id': 'msg-005', 'sender': 'hr@mycompany.com', 'subject': 'Please review and sign the updated policy doc', 'category': 'create_task', 'action': 'create_task', 'status': 'handled'}
{'id': 'msg-006', 'sender': 'deals@shopping-promo.com', 'subject': '50% off everything this weekend only!!!', 'category': 'archive', 'action': 'archive', 's

CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'show'], returncode=0, stdout="\n-- emails --\n{'id': 'msg-001', 'sender': 'alice@customer.com', 'subject': 'Can you send me the Q3 invoice?', 'category': 'reply', 'action': 'reply', 'status': 'handled'}\n{'id': 'msg-002', 'sender': 'no-reply@newsletter.io', 'subject': 'Your weekly digest is here', 'category': 'archive', 'action': 'archive', 'status': 'handled'}\n{'id': 'msg-003', 'sender': 'ops@vendor-systems.com', 'subject': 'URGENT: production outage affecting billing', 'category': 'escalate', 'action': 'escalate', 'status': 'handled'}\n{'id': 'msg-004', 'sender': 'bob@partner.org', 'subject': 'Question about API rate limits', 'category': 'reply', 'action': 'reply', 'status': 'handled'}\n{'id': 'msg-005', 'sender': 'hr@mycompany.com', 'subject': 'Please review and sign the updated policy doc', 'category': 'create_task', 'action': 'create_task', 'status': 'handled'}\n{'id': 'msg-006', 'sender': 'deals@shopping-promo.

## Step 4 — Test cases: automated test suite

This is the formal test suite (`pytest`), separate from the manual demo above:

- `test_sequential_double_run_no_duplicates` -- the baseline case above, automated.
- `test_concurrent_race_no_duplicates` -- 5 separate OS processes racing against the same database at the same instant; exercises the claim-before-act guard (the emails PRIMARY KEY) under real concurrency.
- `test_crash_recovery_no_duplicate_task` -- simulates a mid-action crash, confirms the retry doesn't duplicate the task.
- `test_crash_recovery_reuses_recorded_category_no_reclassify` -- confirms a retried email reuses its originally recorded category instead of reclassifying.
- `test_concurrent_resume_of_crashed_email_no_duplicate` -- a harder version of the concurrency test: races 5 threads (with a barrier, so the race is deterministic) over *resuming* the same crashed email, guarded by a different mechanism (the tasks.email_id UNIQUE constraint) than the initial claim.
- `test_growing_batch_only_new_emails_processed` -- feeds the agent a batch that grows between two runs (new mail alongside already-handled mail), proving per-email state tracking rather than whole-batch detection.
- `test_fetch_new_emails_mailtm_*` / `test_get_or_create_mailtm_account_*` -- verify the real-inbox field-mapping logic against mail.tm's documented schema (mocked HTTP, no live network needed).


In [7]:
subprocess.run([sys.executable, "-m", "pip", "install", "pytest", "-q"])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'pytest', '-q'], returncode=0)

In [8]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "test_agent.py", "test_mailtm_source.py"],
    text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print("--- stderr ---")
    print(result.stderr)

if result.returncode == 0:
    print("\n>>> ALL TESTS PASSED <<<")
else:
    print("\n>>> TESTS FAILED -- scroll up to the FAILED lines and traceback above for the reason <<<")

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/lec
plugins: langsmith-0.11.0, anyio-4.14.2, typeguard-4.6.0
collecting ... collected 9 items

test_agent.py::test_sequential_double_run_no_duplicates PASSED           [ 11%]
test_agent.py::test_concurrent_race_no_duplicates PASSED                 [ 22%]
test_agent.py::test_crash_recovery_no_duplicate_task PASSED              [ 33%]
test_agent.py::test_crash_recovery_reuses_recorded_category_no_reclassify PASSED [ 44%]
test_agent.py::test_concurrent_resume_of_crashed_email_no_duplicate PASSED [ 55%]
test_agent.py::test_growing_batch_only_new_emails_processed PASSED       [ 66%]
test_mailtm_source.py::test_fetch_new_emails_mailtm_maps_fields_correctly PASSED [ 77%]
test_mailtm_source.py::test_fetch_new_emails_mailtm_falls_back_to_html_then_intro PASSED [ 88%]
test_mailtm_s

## Step 5 — Real inbox proof: send test emails from your personal email

This is the live, real-world proof: a real disposable inbox, and real emails you send from your own email address. The pattern below is **send one email, then run its cell** -- repeated several times, so each email you send gets its own permanent, timestamped cell output in this notebook (not overwritten by the next one).

Run the cell below first to get your real inbox address.

In [9]:
run_agent("run", env_overrides={"TRIAGE_SOURCE": "mailtm"})
# Copy the printed address -- you'll send test emails to this one throughout Step 5.

[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com
[mail.tm] send a test email to this address, then run again to see it triaged

=== Run 4776c094 summary ===
  Timestamp:             2026-08-18T17:17:37.576747+00:00
  Source:                mailtm
  Classifier:            heuristic
  Emails fetched:        0
  Newly processed:       0
  Skipped (dup-guard):   0



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com\n[mail.tm] send a test email to this address, then run again to see it triaged\n\n=== Run 4776c094 summary ===\n  Timestamp:             2026-08-18T17:17:37.576747+00:00\n  Source:                mailtm\n  Classifier:            heuristic\n  Emails fetched:        0\n  Newly processed:       0\n  Skipped (dup-guard):   0\n', stderr='')

### Test email #1

Send a real email to the address printed above, from your personal email account. Wait ~10-20 seconds for delivery, then run the cell below. It should show 1 new email, correctly classified.

In [10]:
run_agent("run", env_overrides={"TRIAGE_SOURCE": "mailtm"})

[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com
[mail.tm] send a test email to this address, then run again to see it triaged

=== Run 25c87920 summary ===
  Timestamp:             2026-08-18T17:18:19.707535+00:00
  Source:                mailtm
  Classifier:            heuristic
  Emails fetched:        1
  Newly processed:       1
  Skipped (dup-guard):   0
    - 6a8493d465469a4f1a2c97fa: archived (archive)



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com\n[mail.tm] send a test email to this address, then run again to see it triaged\n\n=== Run 25c87920 summary ===\n  Timestamp:             2026-08-18T17:18:19.707535+00:00\n  Source:                mailtm\n  Classifier:            heuristic\n  Emails fetched:        1\n  Newly processed:       1\n  Skipped (dup-guard):   0\n    - 6a8493d465469a4f1a2c97fa: archived (archive)\n', stderr='')

### Test email #2

Send a **second, different** email to the same address (different subject/body so you can see it get classified differently, e.g. one that sounds urgent). Wait, then run the cell below. It should show the first email correctly skipped as already-handled, and only the new one newly processed -- proving dedup holds even as new real mail keeps arriving.

In [11]:
run_agent("run", env_overrides={"TRIAGE_SOURCE": "mailtm"})

[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com
[mail.tm] send a test email to this address, then run again to see it triaged

=== Run 79a1efc4 summary ===
  Timestamp:             2026-08-18T17:19:07.669119+00:00
  Source:                mailtm
  Classifier:            heuristic
  Emails fetched:        2
  Newly processed:       1
  Skipped (dup-guard):   1
    - 6a8494056ab5522d54c53163: escalated (escalate)
    - 6a8493d465469a4f1a2c97fa: skipped_already_handled



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com\n[mail.tm] send a test email to this address, then run again to see it triaged\n\n=== Run 79a1efc4 summary ===\n  Timestamp:             2026-08-18T17:19:07.669119+00:00\n  Source:                mailtm\n  Classifier:            heuristic\n  Emails fetched:        2\n  Newly processed:       1\n  Skipped (dup-guard):   1\n    - 6a8494056ab5522d54c53163: escalated (escalate)\n    - 6a8493d465469a4f1a2c97fa: skipped_already_handled\n', stderr='')

### Test email #3 (optional -- duplicate this cell pattern for as many as you want)

Send a third test email, wait, then run:

In [ ]:
run_agent("run", env_overrides={"TRIAGE_SOURCE": "mailtm"})

### Final idempotency check

Run the cell below **without** sending any new email first. This should show 0 newly processed and every email you sent skipped as already-handled -- the same guarantee proven on the stub inbox in Step 2, now proven against a real inbox with real emails you sent yourself.

In [12]:
run_agent("run", env_overrides={"TRIAGE_SOURCE": "mailtm"})

[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com
[mail.tm] send a test email to this address, then run again to see it triaged

=== Run 028f1053 summary ===
  Timestamp:             2026-08-18T17:19:21.556005+00:00
  Source:                mailtm
  Classifier:            heuristic
  Emails fetched:        2
  Newly processed:       0
  Skipped (dup-guard):   2
    - 6a8494056ab5522d54c53163: skipped_already_handled
    - 6a8493d465469a4f1a2c97fa: skipped_already_handled



CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'run'], returncode=0, stdout='[mail.tm] polling inbox: triage-agent-97cb64b63b@emalupe.com\n[mail.tm] send a test email to this address, then run again to see it triaged\n\n=== Run 028f1053 summary ===\n  Timestamp:             2026-08-18T17:19:21.556005+00:00\n  Source:                mailtm\n  Classifier:            heuristic\n  Emails fetched:        2\n  Newly processed:       0\n  Skipped (dup-guard):   2\n    - 6a8494056ab5522d54c53163: skipped_already_handled\n    - 6a8493d465469a4f1a2c97fa: skipped_already_handled\n', stderr='')

See everything that was classified from your real inbox, with sender/subject:

In [13]:
run_agent("show", env_overrides={"TRIAGE_SOURCE": "mailtm"})


-- emails --
{'id': 'msg-001', 'sender': 'alice@customer.com', 'subject': 'Can you send me the Q3 invoice?', 'category': 'reply', 'action': 'reply', 'status': 'handled'}
{'id': 'msg-002', 'sender': 'no-reply@newsletter.io', 'subject': 'Your weekly digest is here', 'category': 'archive', 'action': 'archive', 'status': 'handled'}
{'id': 'msg-003', 'sender': 'ops@vendor-systems.com', 'subject': 'URGENT: production outage affecting billing', 'category': 'escalate', 'action': 'escalate', 'status': 'handled'}
{'id': 'msg-004', 'sender': 'bob@partner.org', 'subject': 'Question about API rate limits', 'category': 'reply', 'action': 'reply', 'status': 'handled'}
{'id': 'msg-005', 'sender': 'hr@mycompany.com', 'subject': 'Please review and sign the updated policy doc', 'category': 'create_task', 'action': 'create_task', 'status': 'handled'}
{'id': 'msg-006', 'sender': 'deals@shopping-promo.com', 'subject': '50% off everything this weekend only!!!', 'category': 'archive', 'action': 'archive', 's

CompletedProcess(args=['/usr/bin/python3', 'email_triage_agent.py', 'show'], returncode=0, stdout="\n-- emails --\n{'id': 'msg-001', 'sender': 'alice@customer.com', 'subject': 'Can you send me the Q3 invoice?', 'category': 'reply', 'action': 'reply', 'status': 'handled'}\n{'id': 'msg-002', 'sender': 'no-reply@newsletter.io', 'subject': 'Your weekly digest is here', 'category': 'archive', 'action': 'archive', 'status': 'handled'}\n{'id': 'msg-003', 'sender': 'ops@vendor-systems.com', 'subject': 'URGENT: production outage affecting billing', 'category': 'escalate', 'action': 'escalate', 'status': 'handled'}\n{'id': 'msg-004', 'sender': 'bob@partner.org', 'subject': 'Question about API rate limits', 'category': 'reply', 'action': 'reply', 'status': 'handled'}\n{'id': 'msg-005', 'sender': 'hr@mycompany.com', 'subject': 'Please review and sign the updated policy doc', 'category': 'create_task', 'action': 'create_task', 'status': 'handled'}\n{'id': 'msg-006', 'sender': 'deals@shopping-promo.

## Step 6 (optional) — Real LLM classification

Runs in its own isolated state folder (`llm_demo_state/`) so it never resets or overwrites the results from Steps 2-5 above -- each section stays independent.

In [ ]:
llm_env = {
    "ANTHROPIC_API_KEY": "your-key-here",  # replace with a real key to actually use the LLM path
    "TRIAGE_CLASSIFIER": "llm",
    "STATE_DIR": "llm_demo_state",
}
run_agent("reset", env_overrides=llm_env)
run_agent("run", env_overrides=llm_env)


## Step 7 — Download the evidence

Since state now lives in your mounted Drive folder by default, `state.db`, `sent_replies.log`, and `tasks_created.log` are already saved there permanently -- no download step needed for those, they'll just be sitting in your project folder alongside the code.

The one thing still worth downloading explicitly is this notebook itself, since its cell outputs (the timestamped proof) only get embedded in the `.ipynb` file when you download it: File -> Download -> Download .ipynb.